In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week8-assignment-2"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
# /public/trendytech/datasets/order_data.csv

In [3]:
orders = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/public/trendytech/datasets/order_data.csv")

In [4]:
orders.show(3)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   536378|     null|PACK OF 60 DINOSA...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
|   536378|     null|PACK OF 60 PINK P...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
|   536378|    84991|60 TEATIME FAIRY ...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 3 rows



## aggregate functions

#### simple agg

In [5]:
orders.count() 

541782

In [6]:
orders.select(count("*").alias("rowcount"), countDistinct("InvoiceNo").alias("dist_inv"), sum("Quantity").alias("tot_qty"), avg("UnitPrice").alias("avg_UnitPrice")).show()

+--------+--------+-------+-----------------+
|rowcount|dist_inv|tot_qty|    avg_UnitPrice|
+--------+--------+-------+-----------------+
|  541782|   25858|5175855|4.611565323321928|
+--------+--------+-------+-----------------+



In [7]:
orders.select("InvoiceNo").distinct().count() ### OR countDistinct("InvoiceNo")

25858

In [8]:
orders.select(sum("Quantity"))

sum(Quantity)
5175855


In [9]:
orders.select(avg("UnitPrice"))

avg(UnitPrice)
4.61156532331637


#### selectExpr STYLE

In [10]:
orders.selectExpr(" count(*) as rowcount" , 'count(distinct(InvoiceNo)) as dist_inv', 'sum(Quantity) as tot_qty ', 'avg(UnitPrice)as avg_UnitPrice').show()

+--------+--------+-------+-----------------+
|rowcount|dist_inv|tot_qty|    avg_UnitPrice|
+--------+--------+-------+-----------------+
|  541782|   25858|5175855|4.611565323321929|
+--------+--------+-------+-----------------+



In [11]:
orders.createOrReplaceTempView("orders")

In [12]:
spark.sql("select count(*) as rowcount, count(distinct(InvoiceNo)) as dist_inv, sum(Quantity) as tot_qty , avg(UnitPrice) as avg_UnitPrice from orders ").show()

+--------+--------+-------+-----------------+
|rowcount|dist_inv|tot_qty|    avg_UnitPrice|
+--------+--------+-------+-----------------+
|  541782|   25858|5175855|4.611565323321927|
+--------+--------+-------+-----------------+



#### grouping agg

In [13]:
orders.show(3)

+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
|   536378|     null|PACK OF 60 DINOSA...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
|   536378|     null|PACK OF 60 PINK P...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
|   536378|    84991|60 TEATIME FAIRY ...|      24|01-12-2010 9.37|     0.55|     14688|United Kingdom|
+---------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 3 rows



In [15]:
orders.groupBy("Country","InvoiceNo").agg(sum("Quantity").alias("tot_qty") , sum(expr("Quantity * UnitPrice")).alias("Inv_val")).sort("InvoiceNo").show()

+--------------+---------+-------+------------------+
|       Country|InvoiceNo|tot_qty|           Inv_val|
+--------------+---------+-------+------------------+
|United Kingdom|   536378|    242|192.78000000000003|
|United Kingdom|   536380|     24|              34.8|
|United Kingdom|   536381|    198|449.97999999999996|
|United Kingdom|   536382|    134|430.59999999999997|
|United Kingdom|   536384|    190|             489.6|
|United Kingdom|   536385|     53|            130.85|
|United Kingdom|   536386|    236|508.20000000000005|
|United Kingdom|   536387|   1440|           3193.92|
|United Kingdom|   536388|    108|            226.14|
|     Australia|   536389|    107|            358.25|
|United Kingdom|   536390|   1568|           1825.74|
|United Kingdom|   536392|    103|318.14000000000004|
|United Kingdom|   536393|      8|              79.6|
|United Kingdom|   536394|    544|1024.6800000000003|
|United Kingdom|   536395|    260| 507.8800000000001|
|United Kingdom|   536396|  

In [17]:
orders.groupBy("Country","InvoiceNo").agg( expr("sum (Quantity) as tot_qty") , expr(" sum(Quantity * UnitPrice) as Inv_val   ")  ).sort("InvoiceNo").show()

+--------------+---------+-------+------------------+
|       Country|InvoiceNo|tot_qty|           Inv_val|
+--------------+---------+-------+------------------+
|United Kingdom|   536378|    242|192.78000000000003|
|United Kingdom|   536380|     24|              34.8|
|United Kingdom|   536381|    198|449.97999999999996|
|United Kingdom|   536382|    134|430.59999999999997|
|United Kingdom|   536384|    190|             489.6|
|United Kingdom|   536385|     53|            130.85|
|United Kingdom|   536386|    236|508.20000000000005|
|United Kingdom|   536387|   1440|           3193.92|
|United Kingdom|   536388|    108|            226.14|
|     Australia|   536389|    107|            358.25|
|United Kingdom|   536390|   1568|           1825.74|
|United Kingdom|   536392|    103|318.14000000000004|
|United Kingdom|   536393|      8|              79.6|
|United Kingdom|   536394|    544|1024.6800000000003|
|United Kingdom|   536395|    260| 507.8800000000001|
|United Kingdom|   536396|  

In [19]:
spark.sql("select Country, InvoiceNo, sum (Quantity) as tot_qty, sum(Quantity * UnitPrice) as Inv_val  from orders group by Country, InvoiceNo order by InvoiceNo  ").show()

+--------------+---------+-------+------------------+
|       Country|InvoiceNo|tot_qty|           Inv_val|
+--------------+---------+-------+------------------+
|United Kingdom|   536378|    242|192.78000000000003|
|United Kingdom|   536380|     24|              34.8|
|United Kingdom|   536381|    198|449.97999999999996|
|United Kingdom|   536382|    134|430.59999999999997|
|United Kingdom|   536384|    190|             489.6|
|United Kingdom|   536385|     53|            130.85|
|United Kingdom|   536386|    236|508.20000000000005|
|United Kingdom|   536387|   1440|           3193.92|
|United Kingdom|   536388|    108|            226.14|
|     Australia|   536389|    107|            358.25|
|United Kingdom|   536390|   1568|           1825.74|
|United Kingdom|   536392|    103|318.14000000000004|
|United Kingdom|   536393|      8|              79.6|
|United Kingdom|   536394|    544|1024.6800000000003|
|United Kingdom|   536395|    260| 507.8800000000001|
|United Kingdom|   536396|  

#### windowing agg